## Cell 0: Install

In [1]:
!pip install -q "transformers==4.57.1" "peft==0.18.0" bitsandbytes accelerate "pandas==2.2.2" "pillow==11.3.0" tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.5 MB/s eta 0:00:00


## Cell 1: Kaggle download

In [2]:
import kagglehub
from pathlib import Path

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
COMP_DIR = Path(kagglehub.competition_download("pixels-to-predictions"))
print("Downloaded to:", COMP_DIR)

100%|██████████| 358M/358M [00:03<00:00, 105MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/competitions/pixels-to-predictions


## Cell 2: Imports & Config

In [4]:
import os, json, random, gc, re
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel

from google.colab import drive
drive.mount("/content/drive")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"

BATCH_SIZE_INFER = 4

ADAPTER_DIR = "/content/drive/MyDrive/lora_v2_best"

print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Mounted at /content/drive
GPU: True
Tesla T4


## Cell 3: Load Data

In [5]:
DATA_DIR = COMP_DIR / "data"
if not DATA_DIR.exists():
    DATA_DIR = COMP_DIR

train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

example_name = Path(val_df.iloc[0]["image_path"]).name
matches = list(COMP_DIR.rglob(example_name))

if len(matches) == 0:
    raise FileNotFoundError(example_name)

DATA_ROOT = matches[0].parents[2]

print("DATA_ROOT:", DATA_ROOT)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

DATA_ROOT: /root/.cache/kagglehub/competitions/pixels-to-predictions/images
Train: 3109 | Val: 1048 | Test: 1008


## Cell 4: Helpers

In [6]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def get_candidate_token_ids(tokenizer):
    candidate_ids = {}

    for letter in CHOICE_LETTERS:
        forms = [
            letter,
            " " + letter,
            "\n" + letter,
            letter + ".",
            " " + letter + ".",
        ]

        ids = set()

        for form in forms:
            enc = tokenizer.encode(form, add_special_tokens=False)
            if enc:
                ids.add(enc[-1])

        candidate_ids[letter] = sorted(ids)

    return candidate_ids


def safe_open_image(row):
    path = DATA_ROOT / row["image_path"]

    if not path.exists():
        path = DATA_ROOT.parent / row["image_path"]

    return Image.open(path).convert("RGB")

## Cell 5: Load LoRA v2 model

In [7]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

processor.tokenizer.padding_side = "left"

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

candidate_ids = get_candidate_token_ids(processor.tokenizer)

print("Loaded LoRA v2 checkpoint")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Loaded LoRA v2 checkpoint


## Cell 6: Image variants

In [8]:
def img_224(row):
    return safe_open_image(row).resize((224, 224), Image.BICUBIC)


def img_336(row):
    return safe_open_image(row).resize((336, 336), Image.BICUBIC)


def img_384(row):
    img = safe_open_image(row)
    img.thumbnail((384, 384), Image.BICUBIC)
    return img


def img_512(row):
    img = safe_open_image(row)
    img.thumbnail((512, 512), Image.BICUBIC)
    return img


def img_native(row):
    return safe_open_image(row)

## Cell 7: Prompt variants

In [9]:
def make_choices_text(row):
    return "\n".join(
        f"{CHOICE_LETTERS[i]}. {choice}"
        for i, choice in enumerate(row["choices"])
    )


def prompt_full(row):
    choices_text = make_choices_text(row)
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    prompt = "<image>\n"
    prompt += "You are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"

    if lecture:
        prompt += f"Lecture: {lecture}\n\n"

    if hint:
        prompt += f"Hint: {hint}\n\n"

    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer:"

    return prompt


def prompt_no_lecture(row):
    choices_text = make_choices_text(row)
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))

    prompt = "<image>\n"
    prompt += "You are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"

    if hint:
        prompt += f"Hint: {hint}\n\n"

    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer:"

    return prompt


def prompt_q_only(row):
    choices_text = make_choices_text(row)
    question = clean_text(row.get("question", ""))

    prompt = "<image>\n"
    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer with the correct letter:"

    return prompt


def prompt_q_hint(row):
    choices_text = make_choices_text(row)
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))

    prompt = "<image>\n"

    if hint:
        prompt += f"Hint: {hint}\n\n"

    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer with the correct letter:"

    return prompt


def prompt_q_lecture(row):
    choices_text = make_choices_text(row)
    question = clean_text(row.get("question", ""))
    lecture = clean_text(row.get("lecture", ""))

    prompt = "<image>\n"

    if lecture:
        prompt += f"Lecture: {lecture}\n\n"

    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer with the correct letter:"

    return prompt


def prompt_lecture_cap(row, max_chars=800):
    choices_text = make_choices_text(row)
    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    if lecture and max_chars is not None:
        lecture = lecture[:max_chars]

    prompt = "<image>\n"
    prompt += "You are solving a science multiple-choice question.\n"
    prompt += "Use the image and text to choose the best answer.\n"
    prompt += "Answer with ONLY the letter and the full text of the correct choice.\n\n"

    if lecture:
        prompt += f"Lecture: {lecture}\n\n"

    if hint:
        prompt += f"Hint: {hint}\n\n"

    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer:"

    return prompt

## Cell 8: Prediction function

In [10]:
def predict_with_config(df_batch, img_fn, prompt_fn):
    images = [img_fn(row) for _, row in df_batch.iterrows()]
    prompts = [prompt_fn(row) for _, row in df_batch.iterrows()]

    inputs = processor(
        text=prompts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]

    log_probs = torch.log_softmax(logits, dim=-1)

    preds = []
    margins = []

    for i, (_, row) in enumerate(df_batch.iterrows()):
        scores = []

        for choice_idx in range(len(row["choices"])):
            letter = CHOICE_LETTERS[choice_idx]
            token_ids = candidate_ids[letter]
            score = log_probs[i, token_ids].max().item()
            scores.append(score)

        pred = int(np.argmax(scores))
        preds.append(pred)

        sorted_scores = sorted(scores, reverse=True)
        margin = sorted_scores[0] - sorted_scores[1] if len(sorted_scores) > 1 else 999.0
        margins.append(margin)

    return preds, margins


def run_config(eval_df, img_fn, prompt_fn, batch_size=BATCH_SIZE_INFER, desc="Eval"):
    preds = []
    margins = []

    for start in tqdm(range(0, len(eval_df), batch_size), desc=desc):
        batch = eval_df.iloc[start:start + batch_size]

        try:
            p, m = predict_with_config(batch, img_fn, prompt_fn)
            preds.extend(p)
            margins.extend(m)

        except RuntimeError as e:
            torch.cuda.empty_cache()

            p = []
            m = []

            for j in range(len(batch)):
                single = batch.iloc[j:j+1]
                pp, mm = predict_with_config(single, img_fn, prompt_fn)
                p.extend(pp)
                m.extend(mm)
                torch.cuda.empty_cache()

            preds.extend(p)
            margins.extend(m)

        torch.cuda.empty_cache()

    return preds, margins

## Cell 9: Define experiments

In [11]:
experiments = [
    ("224 + full", img_224, prompt_full),
    ("336 + full", img_336, prompt_full),
    ("384cap + full", img_384, prompt_full),
    ("512cap + full", img_512, prompt_full),
    ("native + full", img_native, prompt_full),

    ("native + no lecture", img_native, prompt_no_lecture),
    ("native + q only", img_native, prompt_q_only),
    ("native + q+hint", img_native, prompt_q_hint),
    ("native + q+lecture", img_native, prompt_q_lecture),

    ("native + lecture cap 800", img_native, lambda row: prompt_lecture_cap(row, 800)),
    ("native + lecture cap 1200", img_native, lambda row: prompt_lecture_cap(row, 1200)),
]

## Cell 10: Run quick 200-sample ablation

In [12]:
RUN_QUICK_ABLATION = True

if RUN_QUICK_ABLATION:
    eval_df = val_df.sample(n=min(200, len(val_df)), random_state=SEED).reset_index(drop=True)

    quick_results = []

    for name, img_fn, prompt_fn in experiments:
        preds, margins = run_config(
            eval_df,
            img_fn,
            prompt_fn,
            desc=name,
        )

        acc = np.mean(np.array(preds) == eval_df["answer"].values)

        quick_results.append({
            "name": name,
            "accuracy": acc,
            "avg_margin": float(np.mean(margins)),
        })

        print(f"{name}: acc={acc:.4f}, avg_margin={np.mean(margins):.4f}")

    quick_results_df = pd.DataFrame(quick_results).sort_values("accuracy", ascending=False)
    display(quick_results_df)

224 + full:   0%|          | 0/50 [00:00<?, ?it/s]

224 + full: acc=0.6700, avg_margin=2.6220


336 + full:   0%|          | 0/50 [00:00<?, ?it/s]

336 + full: acc=0.6550, avg_margin=2.7531


384cap + full:   0%|          | 0/50 [00:00<?, ?it/s]

...

native + lecture cap 800:   0%|          | 0/50 [00:00<?, ?it/s]

native + lecture cap 800: acc=0.6550, avg_margin=2.7505


native + lecture cap 1200:   0%|          | 0/50 [00:00<?, ?it/s]

native + lecture cap 1200: acc=0.6750, avg_margin=2.7518


,name,accuracy,avg_margin
2,384cap + full,0.680,2.764279
10,native + lecture cap 1200,0.675,2.751790
4,native + full,0.675,2.793695
0,224 + full,0.670,2.622024
3,512cap + full,0.660,2.836195
1,336 + full,0.655,2.753112
9,native + lecture cap 800,0.655,2.750519
5,native + no lecture,0.650,3.516823
6,native + q only,0.435,2.292229
8,native + q+lecture,0.380,2.447686


## Cell 11: Run full validation on best configs

In [14]:
RUN_FULL_VAL = True

if RUN_FULL_VAL:
    if "quick_results_df" in globals():
        top_config_names = quick_results_df.head(4)["name"].tolist()
    else:
        top_config_names = [
          "384cap + full",
          "native + lecture cap 1200",
          "native + full",
          "224 + full",
      ]

    print("Running full validation for:", top_config_names)

    full_val_results = []
    val_predictions_by_config = {}

    eval_df = val_df.copy().reset_index(drop=True)

    for name, img_fn, prompt_fn in experiments:
        if name not in top_config_names:
            continue

        preds, margins = run_config(
            eval_df,
            img_fn,
            prompt_fn,
            desc=f"Full val: {name}",
        )

        acc = np.mean(np.array(preds) == eval_df["answer"].values)

        full_val_results.append({
            "name": name,
            "accuracy": acc,
            "avg_margin": float(np.mean(margins)),
        })

        val_predictions_by_config[name] = {
            "preds": preds,
            "margins": margins,
        }

        print(f"{name}: full_val_acc={acc:.4f}, avg_margin={np.mean(margins):.4f}")

    full_val_results_df = pd.DataFrame(full_val_results).sort_values("accuracy", ascending=False)
    display(full_val_results_df)

    full_val_results_df.to_csv("/content/ablation_full_val_results.csv", index=False)

Running full validation for: ['384cap + full', 'native + lecture cap 1200', 'native + full', '224 + full']


Full val: 224 + full:   0%|          | 0/262 [00:00<?, ?it/s]

224 + full: full_val_acc=0.7185, avg_margin=3.0297


Full val: 384cap + full:   0%|          | 0/262 [00:00<?, ?it/s]

384cap + full: full_val_acc=0.7319, avg_margin=3.2551


Full val: native + full:   0%|          | 0/262 [00:00<?, ?it/s]

native + full: full_val_acc=0.7233, avg_margin=3.2815


Full val: native + lecture cap 1200:   0%|          | 0/262 [00:00<?, ?it/s]

native + lecture cap 1200: full_val_acc=0.7261, avg_margin=3.2556


,name,accuracy,avg_margin
1,384cap + full,0.731870,3.255080
3,native + lecture cap 1200,0.726145,3.255562
2,native + full,0.723282,3.281517
0,224 + full,0.718511,3.029673


## Cell 12: Choose best config

In [15]:
BEST_CONFIG_NAME = full_val_results_df.iloc[0]["name"]

print("BEST_CONFIG_NAME:", BEST_CONFIG_NAME)

config_map = {
    name: (img_fn, prompt_fn)
    for name, img_fn, prompt_fn in experiments
}

best_img_fn, best_prompt_fn = config_map[BEST_CONFIG_NAME]

BEST_CONFIG_NAME: 384cap + full


## Cell 13: Analyze wrong predictions

In [16]:
best_preds = val_predictions_by_config[BEST_CONFIG_NAME]["preds"]
best_margins = val_predictions_by_config[BEST_CONFIG_NAME]["margins"]

analysis_df = val_df.copy().reset_index(drop=True)
analysis_df["pred"] = best_preds
analysis_df["margin"] = best_margins
analysis_df["correct"] = analysis_df["pred"] == analysis_df["answer"]

print("Best full-val accuracy:", analysis_df["correct"].mean())

print("\nBy num_choices:")
display(analysis_df.groupby("num_choices")["correct"].agg(["mean", "count"]).sort_values("mean"))

print("\nBy subject:")
display(analysis_df.groupby("subject")["correct"].agg(["mean", "count"]).sort_values("mean"))

print("\nWorst topics:")
display(analysis_df.groupby("topic")["correct"].agg(["mean", "count"]).query("count >= 5").sort_values("mean").head(20))

wrong_df = analysis_df[~analysis_df["correct"]].copy()
display(wrong_df[["id", "question", "choices", "answer", "pred", "margin", "subject", "topic", "skill"]].head(20))

Best full-val accuracy: 0.7318702290076335

By num_choices:


,mean,count
num_choices,,
5,0.250000,44
3,0.690945,508
2,0.795082,244
4,0.837302,252



By subject:


,mean,count
subject,,
language science,0.571429,28
natural science,0.696268,777
social science,0.864198,243



Worst topics:


,mean,count
topic,,
world-history,0.545455,11
chemistry,0.549451,91
writing-strategies,0.571429,21
reading-comprehension,0.571429,7
earth-science,0.592593,81
literacy-in-science,0.600000,5
physics,0.624413,213
biology,0.761905,273
geography,0.806202,129


,id,question,choices,answer,pred,margin,subject,topic,skill
0,val_00671,Why might covering its eggs with its body incr...,"[the leech's eggs will hatch, the leech will n...",0,1,0.734863,natural science,literacy-in-science,How can animal behaviors affect reproductive s...
1,val_04111,Why might fanning eggs increase the reproducti...,[the male will build a nest for females to lay...,1,0,3.687988,natural science,literacy-in-science,How can animal behaviors affect reproductive s...
2,val_02022,"Based on clues in the text, how did fossil evi...",[It helped scientists learn that beetle specie...,3,0,2.078125,language science,reading-comprehension,Read passages about animals
3,val_01237,What is the probability that a Cepaea snail pr...,"[2/4, 4/4, 0/4, 3/4, 1/4]",0,3,0.296875,natural science,biology,Use Punnett squares to calculate probabilities...
4,val_03458,What is the probability that a Cepaea snail pr...,"[2/4, 0/4, 3/4, 1/4, 4/4]",4,3,0.765625,natural science,biology,Use Punnett squares to calculate probabilities...
6,val_03959,What is the expected ratio of offspring that h...,"[1:3, 2:2, 3:1, 0:4, 4:0]",4,0,0.921387,natural science,biology,Use Punnett squares to calculate ratios of off...
7,val_03289,What is the probability that a rock pocket mou...,"[4/4, 2/4, 0/4, 3/4, 1/4]",2,0,0.109375,natural science,biology,Use Punnett squares to calculate probabilities...
10,val_03744,What is the probability that a rose plant prod...,"[4/4, 2/4, 0/4, 1/4, 3/4]",0,3,1.593750,natural science,biology,Use Punnett squares to calculate probabilities...
12,val_02310,What is the probability that a human produced ...,"[4/4, 2/4, 3/4, 0/4, 1/4]",1,4,0.250000,natural science,biology,Use Punnett squares to calculate probabilities...
15,val_03518,What is the expected ratio of offspring with a...,"[2:2, 1:3, 4:0, 0:4, 3:1]",3,0,1.733887,natural science,biology,Use Punnett squares to calculate ratios of off...


## Cell 14: Test inference with best config

In [17]:
all_preds = []
all_margins = []

for start in tqdm(range(0, len(test_df), BATCH_SIZE_INFER), desc=f"Test: {BEST_CONFIG_NAME}"):
    batch = test_df.iloc[start:start+BATCH_SIZE_INFER]

    try:
        p, m = predict_with_config(batch, best_img_fn, best_prompt_fn)
        all_preds.extend(p)
        all_margins.extend(m)

    except RuntimeError:
        torch.cuda.empty_cache()

        for j in range(len(batch)):
            single = batch.iloc[j:j+1]
            p, m = predict_with_config(single, best_img_fn, best_prompt_fn)
            all_preds.extend(p)
            all_margins.extend(m)
            torch.cuda.empty_cache()

    torch.cuda.empty_cache()

print("Predictions:", len(all_preds))
print("Test rows:", len(test_df))

Test: 384cap + full:   0%|          | 0/252 [00:00<?, ?it/s]

Predictions: 1008
Test rows: 1008


## Cell 15: Create submission

In [18]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": all_preds,
})

submission["answer"] = submission["answer"].astype(int)

assert list(submission.columns) == ["id", "answer"]
assert len(submission) == len(sample_submission)
assert submission["id"].tolist() == sample_submission["id"].tolist()
assert submission["answer"].notna().all()

for idx, row in test_df.iterrows():
    pred = int(submission.loc[idx, "answer"])
    assert 0 <= pred < len(row["choices"])

print(submission.head())
print(submission["answer"].value_counts().sort_index())

out_path = "/content/submission.csv"
drive_path = f"/content/drive/MyDrive/submission_lora_v2_ablation_{BEST_CONFIG_NAME.replace(' ', '_').replace('+', 'plus')}.csv"

submission.to_csv(out_path, index=False)
submission.to_csv(drive_path, index=False)

print("Saved:", out_path)
print("Saved:", drive_path)

           id  answer
0  test_01750       1
1  test_00128       0
2  test_02891       0
3  test_02425       3
4  test_00930       1
answer
0    412
1    319
2    203
3     74
Name: count, dtype: int64
Saved: /content/submission.csv
Saved: /content/drive/MyDrive/submission_lora_v2_ablation_384cap_plus_full.csv


## Cell 16: Download submission

In [19]:
from google.colab import files
files.download("/content/submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>